In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 21.386,
	"longitude": 103.023,
	"daily": "weather_code",
	"hourly": ["temperature_2m", "dew_point_2m", "precipitation", "rain", "evapotranspiration", "wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_gusts_10m", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm", "soil_moisture_9_to_27cm", "temperature_80m", "cloud_cover", "surface_pressure", "showers"],
	"timezone": "Asia/Bangkok",
	"start_date": "2026-04-18",
	"end_date": "2026-07-24",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_evapotranspiration = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_wind_speed_80m = hourly.Variables(6).ValuesAsNumpy()
hourly_wind_speed_120m = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_direction_80m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_direction_120m = hourly.Variables(10).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(11).ValuesAsNumpy()
hourly_soil_temperature_0cm = hourly.Variables(12).ValuesAsNumpy()
hourly_soil_temperature_6cm = hourly.Variables(13).ValuesAsNumpy()
hourly_soil_temperature_18cm = hourly.Variables(14).ValuesAsNumpy()
hourly_soil_moisture_0_to_1cm = hourly.Variables(15).ValuesAsNumpy()
hourly_soil_moisture_1_to_3cm = hourly.Variables(16).ValuesAsNumpy()
hourly_soil_moisture_3_to_9cm = hourly.Variables(17).ValuesAsNumpy()
hourly_soil_moisture_9_to_27cm = hourly.Variables(18).ValuesAsNumpy()
hourly_temperature_80m = hourly.Variables(19).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(20).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(21).ValuesAsNumpy()
hourly_showers = hourly.Variables(22).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["evapotranspiration"] = hourly_evapotranspiration
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
hourly_data["soil_temperature_0cm"] = hourly_soil_temperature_0cm
hourly_data["soil_temperature_6cm"] = hourly_soil_temperature_6cm
hourly_data["soil_temperature_18cm"] = hourly_soil_temperature_18cm
hourly_data["soil_moisture_0_to_1cm"] = hourly_soil_moisture_0_to_1cm
hourly_data["soil_moisture_1_to_3cm"] = hourly_soil_moisture_1_to_3cm
hourly_data["soil_moisture_3_to_9cm"] = hourly_soil_moisture_3_to_9cm
hourly_data["soil_moisture_9_to_27cm"] = hourly_soil_moisture_9_to_27cm
hourly_data["temperature_80m"] = hourly_temperature_80m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["showers"] = hourly_showers

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["weather_code"] = daily_weather_code

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Coordinates: 21.335676193237305°N 103.02751922607422°E
Elevation: 482.0 m asl
Timezone: b'Asia/Bangkok'b'GMT+7'
Timezone difference to GMT+0: 25200s

Hourly data
                           date  temperature_2m  dew_point_2m  precipitation  \
0    2026-04-18 00:00:00+07:00             NaN           NaN            NaN   
1    2026-04-18 01:00:00+07:00             NaN           NaN            NaN   
2    2026-04-18 02:00:00+07:00             NaN           NaN            NaN   
3    2026-04-18 03:00:00+07:00             NaN           NaN            NaN   
4    2026-04-18 04:00:00+07:00             NaN           NaN            NaN   
...                        ...             ...           ...            ...   
2347 2026-07-24 19:00:00+07:00       25.640501     24.690500            0.1   
2348 2026-07-24 20:00:00+07:00       25.240501     24.590500            1.1   
2349 2026-07-24 21:00:00+07:00       24.790501     24.290501            1.1   
2350 2026-07-24 22:00:00+07:00       24.390501 

In [3]:
df = hourly_dataframe.copy()
df.head()

,date,temperature_2m,dew_point_2m,precipitation,rain,evapotranspiration,wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_direction_10m,...,soil_temperature_6cm,soil_temperature_18cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,temperature_80m,cloud_cover,surface_pressure,showers
0,2026-04-18 00:00:00+07:00,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-18 01:00:00+07:00,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-18 02:00:00+07:00,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-18 03:00:00+07:00,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-18 04:00:00+07:00,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.describe()

,temperature_2m,dew_point_2m,precipitation,rain,evapotranspiration,wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_direction_10m,wind_direction_80m,...,soil_temperature_6cm,soil_temperature_18cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,temperature_80m,cloud_cover,surface_pressure,showers
count,1889.000000,1889.000000,1889.000000,1889.000000,2352.000000,1889.000000,1889.000000,1889.000000,1889.000000,1889.000000,...,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1822.000000,1889.000000,1889.000000,1889.000000
mean,26.343307,23.615807,0.477660,0.197300,0.098478,4.023005,7.001448,7.331909,174.765640,186.668701,...,27.522118,27.388617,0.351440,0.352635,0.354706,0.358656,26.858097,79.861832,953.520081,0.280360
std,2.520194,1.018479,0.877713,0.321704,0.154815,2.995621,4.035088,4.225540,72.708427,64.409966,...,2.122206,1.140117,0.034943,0.033726,0.031883,0.030077,2.079823,29.632128,2.545161,0.732581
min,21.290501,20.690500,0.000000,0.000000,-0.000000,0.000000,0.350885,0.367446,2.385899,4.085537,...,23.547998,24.547998,0.253000,0.266000,0.276000,0.288000,20.907001,0.000000,946.350769,0.000000
25%,24.340500,22.940500,0.000000,0.000000,0.000000,1.835647,3.847754,4.029364,113.629387,152.102829,...,25.948000,26.647999,0.328000,0.330000,0.335000,0.341000,25.448000,69.000000,951.771912,0.000000
50%,25.940500,23.640501,0.100000,0.100000,0.010000,3.143565,6.357218,6.657273,189.462250,203.962494,...,27.047998,27.448000,0.356000,0.357000,0.358000,0.360000,26.948000,97.000000,953.216309,0.000000
75%,27.890501,24.340500,0.500000,0.300000,0.140000,5.441838,9.514419,9.963490,224.999893,224.999893,...,29.047998,28.147999,0.376000,0.376000,0.377000,0.382000,28.247999,100.000000,954.989014,0.200000
max,33.940502,27.240501,7.400000,2.900000,0.650000,16.774803,26.605431,27.861181,360.000000,360.000000,...,34.698002,30.547998,0.472000,0.471000,0.466000,0.433000,32.748001,100.000000,961.851624,6.900000


In [5]:
print('📋 Basic dataset info displayed')
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-null': df.notnull().sum(),
    'Missing Values': df.isnull().sum(),
    'Unique Values': df.nunique()
})

info_df

📋 Basic dataset info displayed


,Data Type,Non-null,Missing Values,Unique Values
date,"datetime64[s, Asia/Bangkok]",2352,0,2352
temperature_2m,float32,1889,463,230
dew_point_2m,float32,1889,463,111
precipitation,float32,1889,463,56
rain,float32,1889,463,71
evapotranspiration,float32,2352,0,65
wind_speed_10m,float32,1889,463,634
wind_speed_80m,float32,1889,463,1069
wind_speed_120m,float32,1889,463,1066
wind_direction_10m,float32,1889,463,897


In [6]:
df_clean = df.dropna().reset_index(drop=True)
print(f"Removed {len(df) - len(df_clean)} null rows")
df_clean.head()

Removed 652 null rows


,date,temperature_2m,dew_point_2m,precipitation,rain,evapotranspiration,wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_direction_10m,...,soil_temperature_6cm,soil_temperature_18cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,temperature_80m,cloud_cover,surface_pressure,showers
0,2026-05-15 04:00:00+07:00,24.340500,21.040501,0.3,0.2,0.01,0.648999,5.450023,5.707259,213.690094,...,24.047998,25.297998,0.312,0.314,0.322,0.348,25.348000,79.0,951.916138,0.1
1,2026-05-15 05:00:00+07:00,24.290501,21.940500,0.1,0.1,0.01,0.804985,4.574979,4.790914,116.564987,...,23.948000,25.198000,0.312,0.315,0.322,0.348,25.247999,95.0,952.286011,0.0
2,2026-05-15 06:00:00+07:00,24.190500,22.240501,0.0,0.0,0.01,0.900000,3.157964,3.307017,53.130020,...,23.897999,25.098000,0.312,0.315,0.322,0.347,24.848000,88.0,952.741699,0.0
3,2026-05-15 07:00:00+07:00,25.240501,24.190500,0.0,0.0,0.02,2.952219,4.924904,5.157355,127.568665,...,24.147999,24.997999,0.319,0.321,0.326,0.347,24.647999,39.0,953.493347,0.0
4,2026-05-15 08:00:00+07:00,26.640501,24.190500,0.0,0.0,0.05,2.930188,4.035176,4.225633,169.380402,...,24.547998,24.948000,0.319,0.321,0.326,0.346,24.547998,37.0,954.115417,0.0


In [7]:
df_clean.columns

Index(['date', 'temperature_2m', 'dew_point_2m', 'precipitation', 'rain',
       'evapotranspiration', 'wind_speed_10m', 'wind_speed_80m',
       'wind_speed_120m', 'wind_direction_10m', 'wind_direction_80m',
       'wind_direction_120m', 'wind_gusts_10m', 'soil_temperature_0cm',
       'soil_temperature_6cm', 'soil_temperature_18cm',
       'soil_moisture_0_to_1cm', 'soil_moisture_1_to_3cm',
       'soil_moisture_3_to_9cm', 'soil_moisture_9_to_27cm', 'temperature_80m',
       'cloud_cover', 'surface_pressure', 'showers'],
      dtype='str')

In [ ]:
Index(['location_id', 'time', 'temperature_2m', 'dew_point_2m','rain',
       'precipitation', 'wind_speed_10m','wind_gusts_10m',
       'et0_fao_evapotranspiration', 
        'soil_temperature_0_to_7cm','soil_temperature_7_to_28cm',
        'soil_moisture_0_to_7cm','soil_moisture_7_to_28cm',
        'cloud_cover','surface_pressure',

        'y_mua_lon', 'y_sat_lo', 'y_dong_loc',
       'y_mua_da', 'y_lu_lut'],
      dtype='str')

In [9]:
df_son = pd.read_csv('../data/weather_merged_2021_2026_labeled.csv')
df_son.columns

Index(['location_id', 'time', 'temperature_2m (°C)', 'dew_point_2m (°C)',
       'precipitation (mm)', 'surface_pressure (hPa)', 'wind_speed_10m (km/h)',
       'cloud_cover (%)', 'rain (mm)', 'snow_depth (m)', 'snowfall (cm)',
       'wind_gusts_10m (km/h)', 'et0_fao_evapotranspiration (mm)',
       'soil_temperature_0_to_7cm (°C)', 'soil_temperature_7_to_28cm (°C)',
       'soil_moisture_0_to_7cm (m³/m³)', 'soil_moisture_7_to_28cm (m³/m³)',
       'y_mua_lon', 'y_sat_lo', 'y_dong_loc', 'y_mua_da', 'y_lu_lut'],
      dtype='str')

In [8]:
df_clean.to_csv('../data/hourly_weather_meteo_data.csv', index=False)